# 第9回: Human-in-the-Loop

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cosmac-dev/ai-agent-seminar/blob/main/session09/session09_hitl_sbx.ipynb)

AI は大量の情報処理を高速にこなせる一方、曖昧な状況の理解や価値判断、高い影響を伴う最終判断には限界がある。自律性を活かしながらこの限界を補うには、人間の判断を処理フローに組み込む仕組みと、実行環境そのものを隔離する仕組みが必要になる。

第8回、第10回は、次の2つを学ぶ。

- **Human-in-the-Loop（HITL）**: 人間の判断を AI の処理フローへ意図的に組み込む設計パターン
- **サンドボックス**: コマンドに許す権限と資源を制限し、問題が起きても影響範囲を閉じ込める

---

# 0. 環境準備

In [ ]:
# @markdown 実行環境フラグ: Google Colab で実行する場合は True にする
IS_COLAB = False # @param {type:"boolean"}

In [ ]:
if IS_COLAB:
    !git clone https://github.com/cosmac-dev/ai-agent-seminar.git
    %cd ai-agent-seminar/session09

%pip install -q -e ".[server]"
!curl -fsSL https://chatgpt.com/codex/install.sh | CODEX_NON_INTERACTIVE=1 sh

---

## 1. Human-in-the-Loopの概要

Human-in-the-Loop（HITL）は、AI の処理フローへ人間の判断を意図的に組み込む設計パターン。AI は大量の情報処理や案の生成を高速に行える一方、曖昧な状況の理解、価値判断、説明責任、高い影響を伴う最終判断には限界がある。


人間を介在させる条件は、操作の影響度、可逆性、モデルの確信度、組織のポリシーなどから決める。低リスクな読み取り操作まで毎回承認させると、待ち時間とレビュー負荷が増え、承認が形骸化しやすい。

### HITLのユースケース

HITL は、AI の判断だけで処理を完結させることが適切でない場面で利用する。コンテンツモデレーション、カスタマーサポート、金融、法務、生成物の品質管理など、さまざまな領域で「どの判断を自動化し、どの条件で人間へ戻すか」を設計する。

| 活用パターン | 人間の役割 | ユースケース例 |
|---|---|---|
| 承認・拒否 | 高リスクな操作の実行可否を決める | ファイル削除、送信、決済 |
| 修正 | AI の案や操作引数を書き換える | 回答案の編集、保存先の変更 |
| エスカレーション | AI の能力や権限を超える案件を引き取る | 複雑な問い合わせ、例外処理 |
| 意思決定支援 | AI の分析を材料に人間が最終判断する | 審査、診断支援、法務レビュー |
| フィードバック | 評価や訂正を将来の改善に利用する | アノテーション、選好データ収集 |

> **Human-on-the-Loop（HOTL）との違い**: HITL では、特定の判断や操作の前に処理を止め、人間の判断を実行フローへ組み込む。HOTL では、AI が通常は個々の判断で処理を止めずに自律動作し、人間はその動作や結果を監督する（例: ログ監視、品質レビュー）。必要に応じて、実行中の停止や是正、成果物の事後修正、ルールや設定の変更などで介入する。

---

## 2. 承認の実装

### 承認が有効な操作

HITL の中でも、承認が特に有効なのは、エージェントが外部世界へ変更を加える直前。

- ファイルの作成・上書き・削除
- メールやメッセージの送信
- データベースの更新
- 決済、発注、公開、デプロイ
- コマンドや外部ツールの実行

承認時には、操作の内容やリスクに応じて、対象、変更内容、実行理由、想定される影響などを提示し、人間が実行可否を判断できるようにする。

### 承認の設計ポイントと限界

- 承認点はリスクに応じて絞り込む

    人間のレビューは遅く、スケールしにくい。大量の承認要求は「内容を読まずに承認する」状態を招き、人間が承認しても判断そのものを誤ることもある。
- レビューした内容と実行する内容を一致させる

    承認後に引数やプロンプトを組み立て直すと、レビュー対象と実行対象がずれる。承認された値をそのまま後続処理へ渡し、承認後に変更が入らないようにする。
- 承認要求そのものが攻撃面になる

    プロンプトインジェクションで承認を促す文面を作る、無害な要求を大量に混ぜて危険な1件を埋没させる、見た目が似た文字で宛先やパスを偽装するといった手口がある。承認だけに頼らず、許可リストや上限値などの機械的なポリシー検査と併用する。
- 承認が得られない場合の既定動作を決める

    タイムアウトや担当者不在で「既定で許可」に倒すと、承認は形だけになる。既定は保留または拒否とする。また、中断中も外部の状態は変化するため、待ち時間が長い承認には有効期限を設け、再開時に前提を検証する。

### LangGraphによる承認フロー

LangGraph の `interrupt()` はノードの実行を一時停止し、レビューに必要な情報を呼び出し側へ返す。人間の判断を `Command(resume=...)` で渡すと、同じ `thread_id` のチェックポイントから処理を再開できる。

ここでは、ファイルへの書き込み案を作り、承認された場合だけ `apply_write` ノードへ進む。ノードへ到達したことは完了メッセージで確認し、実際のファイル書き込みは行わない。


LLM は使わず、承認制御だけに注目する。実際のエージェントでは、最初の「書き込み案」を LLM やツール呼び出しが生成する。また、`apply_write`では実際に書き込まずに完了メッセージを表示して処理を終了する。

[![](https://mermaid.ink/img/pako:eNp1Uk1PwkAQ_SvNnGpSCGD6tQcTIx71oJxkCdm0A622u82wBZHy390Wqa3CnObtvjf73mQPEKkYgcEqU7soEaSt2ZQTl5apjTbYnr_O7l9mi5vzaUFYCMLljlKNdg-1nKTMhVwSblPc2V3QMkRRZPufGZ2-vSeMFMVG9Y6RTpW0_x60zCXK2J4_Pk9PFnvmrcHgrm_4YoqG1rV5KUdDqjjMa--ktmhVFsapthYcqm6e6-KT-epfuAtLaQR1tGsL6RF-Q-8z7D-8SrOMEbY0cGBNaQxMU4kO5Ei5qCEc6hEcdII5cmCmjQV9cODyaDSFkG9K5WcZqXKdnEFZxELjNBVrEoaxEtmmphhvSA-qlBpYeOs3M4Ad4NPAcDhyA3ccemM_cEe-58AemDsZhqb8iRd6wdgNJ0cHvppHR8PAdx2o163o6fRjm497_AZhue6u?type=png)](https://mermaid.live/edit#pako:eNp1UstOwzAQ_JVoT0FKqz6U5wEJUY5wgJ6oq8qKt40hiaPFaSlN_x0npWkCZU879sx6ZuUDxEogRLBO1S5OOGlrPmPEcsvUhzbYXrzM757ny5vzaUFYcMLVjqRGu4daTlJmPF8RbiXu7C5oGbwo0v3PjE7f3hPGioRRvWGspcrt3wctc4W5sBcPT7OTxZ55azC47Ru-mqKhdW1ey9GQKgaL2jupLVqVhULqJYOqG-d_7cl79SfblZ00gjrZf_voES6Z9yn2H17LNI0IWxo4sCEpINJUogMZUsZrCId6BAOdYIYMItMKTu8MWH40moLnr0plZxmpcpOcQVkIrnEm-Yb4hWGsId2rMtcQhdNpMwKiA3waGA5HbuCOQ2_sB-7I9xzYQ-ROhqEpf-KFXjB2w8nRga_mzdEw8F0H6mUrejz91-bbHr8Bzu_uPw)

In [ ]:
import json
from typing import Literal, TypedDict
from uuid import uuid4

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

# State
class FileApprovalState(TypedDict, total=False):
    path: str
    content: str
    approved_content: str
    decision: Literal['approve', 'reject']
    review_note: str
    result: str

# Nodes
def prepare_write(state: FileApprovalState) -> dict:
    if not state['content']:
        raise ValueError('空の内容は書き込めません')
    return {'approved_content': state['content']}

def human_review(state: FileApprovalState) -> dict:
    # interrupt より前では副作用を起こさない
    response = interrupt(
        {
            'action': 'write_file',
            'path': state['path'],
            'preview': state['content'],
            'allowed_decisions': ['approve', 'edit', 'reject'],
        }
    )

    decision_type = response.get('type')
    if decision_type == 'approve':
        return {
            'decision': 'approve',
            'approved_content': state['content'],
            'review_note': '提案どおり承認',
        }
    if decision_type == 'edit':
        edited_content = response.get('content')
        if not isinstance(edited_content, str) or not edited_content:
            raise ValueError('edit では空でない content が必要です')
        return {
            'decision': 'approve',
            'approved_content': edited_content,
            'review_note': '人間が内容を修正して承認',
        }
    if decision_type == 'reject':
        return {
            'decision': 'reject',
            'review_note': str(response.get('message', '理由なし')),
        }
    raise ValueError(f'未対応の判断です: {decision_type}')


def route_after_review(state: FileApprovalState) -> str:
    return 'record_rejection' if state['decision'] == 'reject' else 'apply_write'


def apply_write(state: FileApprovalState) -> dict:
    return {'result': f'書き込み完了: {state['path']}'}


def record_rejection(state: FileApprovalState) -> dict:
    return {'result': f"書き込み中止: {state['review_note']}"}

In [ ]:
approval_builder = StateGraph(FileApprovalState)
approval_builder.add_node('prepare_write', prepare_write)
approval_builder.add_node('human_review', human_review)
approval_builder.add_node('apply_write', apply_write)
approval_builder.add_node('record_rejection', record_rejection)

approval_builder.add_edge(START, 'prepare_write')
approval_builder.add_edge('prepare_write', 'human_review')
approval_builder.add_conditional_edges(
    'human_review',
    route_after_review,
    {
        'apply_write': 'apply_write',
        'record_rejection': 'record_rejection',
    },
)
approval_builder.add_edge('apply_write', END)
approval_builder.add_edge('record_rejection', END)

# 中断・再開にはチェックポインタが必要
approval_graph = approval_builder.compile(checkpointer=InMemorySaver())

In [ ]:
approval_config = {
    'configurable': {'thread_id': f'file-write-{uuid4()}'}
}
proposal = {
    'path': 'reports/summary.md',
    'content': '# 実行結果\n\n承認後に書き込む想定の内容です。\n',
}

# human_review で停止する
pending = approval_graph.invoke(proposal, config=approval_config)
review_request = pending['__interrupt__'][0].value
print(json.dumps(review_request, ensure_ascii=False, indent=2))

In [ ]:
# 同じ thread_id に人間の判断を渡して再開する
approved = approval_graph.invoke(
    Command(resume={'type': 'approve'}),
    config=approval_config,
)

print(approved['result'])

In [ ]:
# 別スレッドで拒否の経路も確認する
reject_config = {
    'configurable': {'thread_id': f'file-write-{uuid4()}'}
}
rejected_proposal = {
    'path': 'reports/delete-me.md',
    'content': 'この内容は保存されません。\n',
}

# 実行
approval_graph.invoke(rejected_proposal, config=reject_config)

# 再開
rejected = approval_graph.invoke(
    Command(
        resume={
            'type': 'reject',
            'message': '保存先と内容を再確認するため',
        }
    ),
    config=reject_config,
)

print(rejected['result'])

### 実装の要点

- **`interrupt()`**: JSON シリアライズ可能なレビュー情報を返し、グラフを停止する
- **`InMemorySaver`**: `thread_id` ごとの状態を保持する。教材には便利だが、プロセス終了で消えるため、本番ではデータベース等の永続チェックポインタへ置き換える
- **`Command(resume=...)`**: 人間の判断を `interrupt()` の戻り値として渡し、同じスレッドを再開する
- **副作用の分離**: `human_review` ノードは停止・再開時に先頭から再実行されるため、副作用を伴う実処理は後続ノードに置く。このデモの `apply_write` は完了メッセージだけを返す

`edit` を試す場合は、再開時に `{'type': 'edit', 'content': '修正後の内容'}` を渡す。人間が編集した内容が `approved_content` として固定され、後続ノードへ渡される。

---

### ReActグラフのリファクタリング

これまでは、モデルノード、ツールノード、条件分岐を接続し、ReAct ループを自前で組み立てた。ノードと辺を直接書く方法は任意のワークフローを表現できるが、その柔軟性が生きるのは構造そのものを設計する場合。ReAct のように構造が標準化されたループでは、ツール呼び出しの検出、実行結果のメッセージ履歴への追加、ループ終了の判定がどれも定型処理になり、自前で書いても得られる利点は小さい。記述量が増え、実装ごとの差異や抜け漏れが入り込む余地だけが残る。

ここからはコードの見通しをよくするため、ReAct ループの構築を `create_agent()` に任せる。`create_agent()` は、コンパイル済みのReActループグラフを生成する。

In [ ]:
# @title create_agentによるReActグラフの生成
from IPython.display import Image, display
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.runnables.graph_mermaid import CurveStyle
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel

@tool
def fake_tool(input: str) -> str:
    """ダミーのツール"""
    return f'fake_tool: {input}'

fake_model = FakeMessagesListChatModel(responses=[])

react_agent = create_agent(
    model=fake_model,
    tools=[fake_tool],
)

display(
    Image(
        react_agent.get_graph().draw_mermaid_png(curve_style=CurveStyle.NATURAL)
    )
)

#### Middleware（フック）の仕組み

`create_agent()` は、モデルやツールの前後へ処理を差し込む Middleware を受け取る。Middleware は `before_agent`、`after_agent`、`before_model`、`after_model`、`wrap_model_call`、`wrap_tool_call` などのフックを通じて、プロンプトの変更、ログ記録、再試行、ツール実行の制御といった横断的な処理を ReAct ループへ追加する。

[![](https://mermaid.ink/img/pako:eNqdVFFvmzAQ_ivW7aWVSBTSkARrmrQ2fdwe1jxtTJEbTIJicGSM2izKf98BgRyMVO14AO78fd_5u5N9hLUOJXCIlH5Zb4WxbLkITJAyfJ5lpI1cJYhQNwHQMIDbGpVZZN38CuBp-fXHMoDfzUpN7DKs1irDfPkl-ZVMw0Lo8fuCypzrio1M7WUbZUjIIrLSNCASEUwbW2-PRGdsYy1_3hix37IXfFeI1VooVa83JusEGrjOL-w2dCoRxkaubaxTdr-k-ao_benaRqtMMQE2GHxptaprmq5RcMtA3-gRjOgrLeglfL4wWqYpoaNXFaGj6MM3aucSb-DJErtAeztz7UsopcSqMwM6C5rL7EHJfwyaV-6OHXMo3ggxeicHocjwzBlx4MxjXg__YpjS38FuzaMX77AoVoqzT5PJpJdbGf8gl7b9v6gfrKridPdU0u8cjyXC7KQZ4Jx4qlPZHRI4sDFxCNyaXDqQSJOIIoRjhcQraSsTGQDH3xC1AigWTsjbi_Sn1klNNTrfbOsg34fCykUs8LAjIhIqKyC4C2kedJ5a4BP3zitFgB_hFbjvD0fe3HP9qTube6PZ1IEDcG889PGZjaf-dO56_vjkwJ-y6mg4n6GADGOrzbfqyi5v7tNf3f3eqw?type=png)](https://mermaid.live/edit#pako:eNqdVFFvmzAQ_ivW7aWVSBTSkARrmrQ2fdwe1jxtTJEbTIJicHQYtVmU_74DAjGMVO38YLi777vzdyf7CGsdSuAQKf2y3go0bLkIMEgZrWcZaZSrhBDqJgDbDOC2RmWGWDe_Anhafv2xDOB3E6mJXYbRWmXkL7-WfyXTsEj0-H1hpznXFRuZmssxStMii8hIbECWZWHa2Pp4lnXGNtLy5w2K_Za90F4hVmuhVB1vRNYOEnCdX8ht6HaKMEa5NrFO2f3S9lf9aaeuZbTKFBNgg8GXVqu6ou2YDW4J6Bs9gQl9pQW9hM8XRku0Tejkq4rYo-jDN9nOJd7AWyF2gfZ25trXopQpVp0Z2LOwfZk5KPmPQHzl7tjBQ7ETBPVODkKR0Z1DceDMY14P_yLYpr-D3ZpHL95hUawUZ58mk0kvtxL-Qa7d9v-ifrCqitPdU0m_czyWCNxJHNCceKpT2R0SOLDBOARuMJcOJBITUZhwrJD0JG1lIgPg9BtSrgCKwIl4e5H-1DqpqajzzRZ4JFRGVr4PhZGLWNBtTxov0jEkPug8NcAn7t2ozAL8CK_AfX848uae60_d2dwbzaYOHIB746FPazae-tO56_njkwN_yrKj4XzmOSDD2Gj8Vr3Z5dN9-gu7l97x)

`HumanInTheLoopMiddleware` は `after_model` フックを利用する。モデルが返した `AIMessage` のツール呼び出しを調べ、`interrupt_on` の対象であれば、`tools` ノードへ進む前に `interrupt()` でグラフを中断する。再開時に渡された `approve`、`edit`、`reject` の判断をツール呼び出しへ反映してから処理を続ける。

今回は、外部へ変更を加える `write_file` を承認対象にし、読み取り専用の `read_file` はそのまま実行する。

In [ ]:
# @title HumanInTheLoopMiddlewareの設定
from langchain.agents.middleware import HumanInTheLoopMiddleware

hitl_middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        'write_file': {
            'allowed_decisions': ['approve', 'edit', 'reject'],
        },
        'file_delete': {
            'allowed_decisions': ['approve', 'edit', 'reject'],
        },
        'read_file': False,
    },
    description_prefix='Toolの実行には承認が必要',
)

In [ ]:
# @title APIキーの設定
import getpass
import os

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OPENAI_API_KEY を入力する: ')

# guarded_agent の web_search / fetch_url は import 時に Tavily を初期化する
if not os.environ.get('TAVILY_API_KEY'):
    os.environ['TAVILY_API_KEY'] = getpass.getpass('TAVILY_API_KEY を入力する: ')

print('OpenAI APIキー設定完了' if os.environ.get('OPENAI_API_KEY') else 'OpenAI APIキー未設定')
print('Tavily APIキー設定完了' if os.environ.get('TAVILY_API_KEY') else 'Tavily APIキー未設定')

#### 長期記憶グラフへ組み込む

- 長期記憶グラフ(`load_memory`, `extract_memory`, `validate_memory`, `write_memory`)に `create_agent()`が返したグラフをノード（サブグラフ）として追加する。
- 外側の長期記憶グラフと内側のReActグラフは状態を共有する必要がある
- `load_memory`で取得した情報をReActの`model`のプロンプトに差し込む(`dynamic_prompt`)


[![](https://mermaid.ink/img/pako:eNp9lFtvmzAUx78Kcl86iURATEPQNKlaJm1Su4c0TysV8rCTWPEFOWYJi_Lda5xCDMlmicuxfz6XPz4cQSExASlYMbkvNkhp72mRqUx4Zuy0se9fM_CyfFwsM_D2qV1hEuGcEy5VfZ8Bx8pAx5CDVqjQF6w_4ZB_EKMYaXJBBzMOu1fUBV3ToXIicJP4t5_zNu2uqOr3WqFy4y3IY6Hb2WZgqkihqRTecu7OWxlyRUzuN8VoBjcqMpOPfTqJNENLyXZmzT4HaybPi2cn2y52m-3e3HPrPS8QY94rrgXitMhLJTkv9XgAvLleugwHoV3TfUcrTVTe1vS94kj8EMsNeZKyfKYYM7JHiowdbFDWP-TzRqMvw0pcdFhkg7tBhqJ6n_9DDJ1Z9vwNbpdqw3UfpDvGF5l6fWFp9-Rft4ZFesfMGuc4_Wa42TMWHLbC7Z75UJZece6chfKPgq6Lqhm5Ek0d0jDyVd3cDaLkloww2pkfhUJ16sVe3N_vynkT970VZSz17iCE_a1nbezqXRSZcE7sTlwqti-WjvyJx5HaEjUyBaVCCuJW1FzAB2tFMUi1qogPOFEcNSY4Npzpxw3hJAOpecXGUwYycTJ7SiR-ScnbbUpW601rVGWj-Zwi05SGWCG2axCTAVFfZSU0SMN4Zn2A9AgOxgzjMUzgNJ4mwRRO4MSs1mY6gONZHAUJTKIkCOPw5IO_NmowTh4iOEsCCGcP04c4hD4gmGqpns9_avvDPr0DX3XW_A?type=png)](https://mermaid.live/edit#pako:eNp9lF1vmzAUhv8Kcm86iURATEPQNKlaJm1Su4s0VysV8rCTWPEHcswSFuW_1ziFGJLNUgLn-PH5eOFwBIXEBKRgxeS-2CClvadFpjLhmbXTxr5_zcDL8nGxzMDbp3aHSYRzTrhU9X0GHCsDHUMOWqFCX7C-wyH_IEYx0uSCDjwOu1fUBV3ToXIicFP4t5_ztuyuqer3WqFy4y3IY6Fbb7MwVaTQVApvOXf9VoZcEVP7TTGaxY2KzNRjr04hzdJSsp3Zs9fBnqnzEtmptsvdVrs3_7mNnheIMe8V1wJxWuSlkpyXejwA3twoXYWD1K7p3qOVJipve_pecSR-iOWGPElZPlOMGdkjRcYONmjrH_J5o9GXYScuOmyywd0kQ1G9z_8hhsEse34Gt1u16boH0r3GF5l6c2Fp982_Hg2L9F4za5zz9Ifh5sxYcDgKt2fmQ1l6xbk-C-UfDV03VTNyJZo6pGHkq7r5N4iSWzLCaGc-FArVqRd7cf-8K-dN3PdWlLHUu4MQ9o-etbG7d1Fk0jm5O3Gp2L5YOvInHkdqS9TINJQKKYjbUfMDPlgrikGqVUV8wIniqDHBseHMPG4IJxlIzS02kTKQiZM5UyLxS0reHlOyWm9AukJsZ6yqbESfU2SmkndeZUog6qushAZpGEMbBKRHcDBmGI9hAqfxNAmmcAInMx_Uxh3A8SyOggQmURKEcXjywV-bNhgnDxGcJQGEs4fpQxyacARTLdXz-VNtv9indzZm10I)

In [ ]:
# @title 長期記憶グラフの状態と読み込みノード
from dataclasses import dataclass
from typing_extensions import NotRequired

from langchain.agents.middleware import AgentState
from langchain.messages import HumanMessage
from langgraph.runtime import Runtime


class LongTermMemoryState(AgentState):
    memories: NotRequired[list[str]]
    memory_candidates: NotRequired[list[str]]
    approved_memories: NotRequired[list[str]]

In [ ]:
# @title create_agentのグラフを長期記憶グラフへ追加
from langchain.agents.middleware import ModelRequest, dynamic_prompt
from langchain_openai import OpenAIEmbeddings
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.store.memory import InMemoryStore
from langchain_openai import ChatOpenAI

from guarded_agent.state import Context
from guarded_agent.memory import load_memory, make_extract_memory, validate_memory, write_memory
from guarded_agent.tools import DEFAULT_TOOLS


@dynamic_prompt
def memory_system_prompt(request: ModelRequest) -> str:
    memories = request.state.get('memories', [])
    prompt = """あなたはAIエージェントです。
必要な場合はToolを利用してください。
"""
    if memories:
        memory_text = '\n'.join(f'- {memory}' for memory in memories)
        prompt += f"""
参考になる長期記憶:
{memory_text}
"""
    return prompt

model_id = 'gpt-5.4-nano' #@param ['gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-5.6-luna', 'gpt-5.5', 'gpt-5.4', 'gpt-5.4-mini', 'gpt-5.4-nano']
model = ChatOpenAI(model=model_id)
memory_hitl_agent = create_agent(
    model=model,
    tools=DEFAULT_TOOLS,
    state_schema=LongTermMemoryState,
    middleware=[memory_system_prompt, hitl_middleware],
)

long_term_store = InMemoryStore(
    index={
        'embed': OpenAIEmbeddings(model='text-embedding-3-small'),
        'dims': 1536,
        'fields': ['text'],
    }
)

builder = StateGraph(
    LongTermMemoryState,
    context_schema=Context,
)
builder.add_node('load_memory', load_memory)
builder.add_node('agent', memory_hitl_agent)
builder.add_node('extract_memory', make_extract_memory(model))
builder.add_node('validate_memory', validate_memory)
builder.add_node('write_memory', write_memory)

builder.add_edge(START, 'load_memory')
builder.add_edge('load_memory', 'agent')
builder.add_edge('agent', 'extract_memory')
builder.add_edge('extract_memory', 'validate_memory')
builder.add_edge('validate_memory', 'write_memory')
builder.add_edge('write_memory', END)

graph_with_hitl_tools = builder.compile(
    store=long_term_store,
    checkpointer=InMemorySaver(),
)

display(
    Image(
        graph_with_hitl_tools.get_graph(xray=2).draw_mermaid_png(curve_style=CurveStyle.NATURAL)
    )
)

#### ツール呼び出しを承認する

書き込みを依頼すると、モデルが生成した `write_file` のツール呼び出しを `HumanInTheLoopMiddleware` の `after_model` フックが検出し、ツールの実行前にグラフを中断する。中断情報にはツール名と引数が含まれるため、実行内容を確認してから判断を返す。

ここでは `approve` を返して提案どおり実行する。引数を変更する場合は `edit`、実行しない場合は `reject` を選ぶ。いずれも中断時と同じ `thread_id` で再開する。

In [ ]:
# @title 共通設定
import json
from langgraph.types import Command

hitl_context = Context(user_id='session09-user')


In [ ]:
# @title write_fileの実行
hitl_config = {
    'configurable': {'thread_id': f'hitl-agent-{uuid4()}'}
}

pending_tool_call = graph_with_hitl_tools.invoke(
    {
        'messages': [
            HumanMessage(
                content=(
                    '私の好みは簡潔な回答です。'
                    'この内容をprofile.txtへ保存してください。'
                )
            )
        ]
    },
    config=hitl_config,
    context=hitl_context,
)

review = pending_tool_call['__interrupt__'][0].value
print(json.dumps(review, ensure_ascii=False, indent=2))


In [ ]:
# @title 承認してグラフを再開する
completed_tool_call = graph_with_hitl_tools.invoke(
    Command(
        resume={
            'decisions': [{'type': 'approve'}],
        }
    ),
    config=hitl_config,
    context=hitl_context,
)

completed_tool_call['messages'][-1].pretty_print()

saved_memories = long_term_store.search(
    ('memories', hitl_context.user_id)
)
print('保存された長期記憶:', [item.value['text'] for item in saved_memories])


In [ ]:
# @title file_deleteの実行
from guarded_agent.tools import WORKSPACE_DIR

delete_hitl_config = {
    'configurable': {'thread_id': f'hitl-agent-{uuid4()}'}
}

pending_delete = graph_with_hitl_tools.invoke(
    {'messages': [HumanMessage(content='profile.txtを削除してください。')]},
    config=delete_hitl_config,
    context=hitl_context,
)

review = pending_delete['__interrupt__'][0].value
print(json.dumps(review, ensure_ascii=False, indent=2))

rejected_delete = graph_with_hitl_tools.invoke(
    Command(
        resume={
            'decisions': [{
                'type': 'reject',
                'message': '保存済みのプロファイルは削除しない',
            }],
        }
    ),
    config=delete_hitl_config,
    context=hitl_context,
)

rejected_delete['messages'][-1].pretty_print()

profile_path = WORKSPACE_DIR / 'profile.txt'
print('profile.txt は残っている:', profile_path.exists())
